<a href="https://colab.research.google.com/github/Joaoplims/sna_roblox_kg/blob/main/Roblox_KG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
pip install requests

## SETUP

In [2]:
from google.colab import userdata

API_KEY = userdata.get('roblox_oauth')
HEADERS = {
    "x-api-key": API_KEY,
    "Content-Type": "application/json"
}

## Utilidades

In [14]:
def save_df_to_csv(df, filename="data_export.csv"):
    """
    Salva um DataFrame do pandas em um arquivo CSV.
    """
    try:
        df.to_csv(filename, index=False, encoding='utf-8-sig')
        print(f"✅ Arquivo '{filename}' salvo com sucesso!")
    except Exception as e:
        print(f"❌ Erro ao salvar o arquivo: {e}")

# Exemplo de uso com os dados processados anteriormente:
# save_df_to_csv(df_users, "usuarios_roblox.csv")

## Função de Chamada para a API

In [19]:
import requests
import time
import random

def make_roblox_request(url, method="GET", params=None, data=None, max_retries=5, auto_paginate=True):
    """
    Função genérica para realizar chamadas à API do Roblox com suporte a Rate Limiting e Paginação.
    """
    all_data = []
    current_params = params.copy() if params else {}

    while True:
        retry_count = 0
        success = False

        while retry_count <= max_retries:
            try:
                response = requests.request(
                    method=method,
                    url=url,
                    headers=HEADERS,
                    params=current_params,
                    json=data
                )

                if response.status_code == 200:
                    res_json = response.json()
                    # Se o resultado for uma lista ou tiver uma chave 'data'
                    page_items = res_json.get('data', []) if isinstance(res_json, dict) else res_json

                    if isinstance(page_items, list):
                        all_data.extend(page_items)
                    else:
                        return res_json # Retorno direto se não for lista

                    # Lógica de Paginação
                    next_cursor = res_json.get('nextPageCursor')
                    if auto_paginate and next_cursor:
                        current_params['cursor'] = next_cursor
                        print(f"📄 Página processada. Buscando próxima página...")
                        success = True
                        break # Sai do loop de retry para a próxima página
                    else:
                        print(f"✅ Sucesso [{method}]: {url} (Total itens: {len(all_data)})")
                        return {"data": all_data} if isinstance(res_json, dict) else all_data

                elif response.status_code == 429:
                    retry_count += 1
                    wait_time = int(response.headers.get("retry-after", (2 ** retry_count) + random.uniform(0, 1)))
                    print(f"⚠️ Rate Limited (429). Aguardando {wait_time}s...")
                    time.sleep(wait_time)
                    continue
                else:
                    print(f"❌ Erro {response.status_code}: {response.text}")
                    return None
            except Exception as e:
                print(f"❌ Erro na requisição: {e}")
                return None

        if not success: break

    return None

# --- Teste usando a função com paginação ---
# Exemplo: Buscar membros com limite pequeno mas forçando múltiplas páginas
group_id_test = 4705120
members_paginated_url = f"https://groups.roblox.com/v1/groups/{group_id_test}/users"
params = {"limit": 100, "sortOrder": "Asc"}

# Limitamos a paginação manual no print para o exemplo não ficar gigante
print("Buscando membros com paginação...")
results = make_roblox_request(members_paginated_url, params=params, auto_paginate=False) # Mude para True para pegar TUDO
if results:
    print(f"Itens na primeira página: {len(results.get('data', []))}")
    print(f"Itens na primeira página: {results.get('data', [])}")

Buscando membros com paginação...
✅ Sucesso [GET]: https://groups.roblox.com/v1/groups/4705120/users (Total itens: 100)
Itens na primeira página: 100
Itens na primeira página: [{'user': {'hasVerifiedBadge': True, 'userId': 947838656, 'username': 'Scriptbloxian', 'displayName': 'Scriptbloxian'}, 'role': {'id': 31527786, 'name': 'Lead Developer', 'rank': 255, 'color': 0}}, {'user': {'hasVerifiedBadge': False, 'userId': 1027961385, 'username': 'Iamsmarterthanyahya3', 'displayName': 'Iamsmarterthanyahya3'}, 'role': {'id': 31527788, 'name': 'Legend', 'rank': 1, 'color': 0}}, {'user': {'hasVerifiedBadge': False, 'userId': 357829182, 'username': 'FallenGh', 'displayName': 'FallenGh'}, 'role': {'id': 31527788, 'name': 'Legend', 'rank': 1, 'color': 0}}, {'user': {'hasVerifiedBadge': False, 'userId': 244267273, 'username': 'ethanwolam', 'displayName': 'kidsgivemesloppy'}, 'role': {'id': 31527788, 'name': 'Legend', 'rank': 1, 'color': 0}}, {'user': {'hasVerifiedBadge': False, 'userId': 740396998,

## Encontrar ID Seed Group

In [7]:
import time
# Cache do id do grupo mais popular (Scriptbloxian Studios) para evitar chamadas excessiva
ID_SEED_GROUP = 0
# Buscar o Grupo 'Scriptbloxian Studios'
# Pesquisas indicam que é um dos grupos mais populares (https://roblox.fandom.com/pt-br/wiki/Scriptbloxian_Studios)
# Grupo de desenvolvedor de jogos famoso
group_search_url = "https://groups.roblox.com/v1/groups/search/lookup?groupName=Scriptbloxian%20Studios"
group_search_results = make_roblox_request(group_search_url)

if group_search_results and group_search_results.get('data'):
    group_info = group_search_results['data'][0]
    group_id = group_info['id']
    ID_SEED_GROUP = group_id
    print(f"🎯 Grupo Encontrado: {group_info['name']} (ID: {group_id})")
else:
    print("❌ Grupo não encontrado ou erro na busca.")

✅ Sucesso [GET]: https://groups.roblox.com/v1/groups/search/lookup?groupName=Scriptbloxian%20Studios
🎯 Grupo Encontrado: Scriptbloxian Studios (ID: 4705120)


## Consulta de usuários

In [11]:
SAMPLE_SIZE = 25  # Parametrizável

# 2. Extrair membros do grupo semente
members_url = f"https://groups.roblox.com/v1/groups/{ID_SEED_GROUP}/users?sortOrder=Asc&limit={SAMPLE_SIZE}"
members_data = make_roblox_request(members_url)

user_ids = []
if members_data and 'data' in members_data:
    user_ids = [member['user']['userId'] for member in members_data['data']]
    print(f"👥 Extraídos {len(user_ids)} usuários do grupo semente (ID: {ID_SEED_GROUP}).")
    print(f"Amostra de IDs: {user_ids}")
else:
    print("❌ Não foi possível extrair os membros do grupo.")

✅ Sucesso [GET]: https://groups.roblox.com/v1/groups/4705120/users?sortOrder=Asc&limit=25
👥 Extraídos 25 usuários do grupo semente (ID: 4705120).
Amostra de IDs: [947838656, 1027961385, 357829182, 244267273, 740396998, 731822913, 871184693, 262953699, 940534557, 970378427, 386824609, 747056849, 452079889, 519533783, 174791020, 363117394, 1041550312, 1038182669, 727787047, 557051148, 1023552809, 112492866, 363983769, 1014686408, 726703596]


### Processamento de Dados dos Usuários
Nesta etapa, consultamos a API de Usuários da Cloud API para obter detalhes como `createTime` e `locale`, calculando métricas derivadas.

In [13]:
from datetime import datetime, timezone

# Parâmetros
MIN_YEARS_FOR_ADULT_INFERENCE = 10
YEARS_FOR_INTERMEDIATE = 4
processed_users = []

now = datetime.now(timezone.utc)

print(f"🔍 Iniciando processamento de {len(user_ids)} usuários...\n")

for uid in user_ids:
    # Endpoint da Cloud API v2 para usuários
    url = f"https://apis.roblox.com/cloud/v2/users/{uid}"
    data = make_roblox_request(url)

    if data:
        create_time_str = data.get('createTime')
        create_time_dt = datetime.fromisoformat(create_time_str.replace('Z', '+00:00'))

        # Cálculo do tempo de conta em anos
        delta = now - create_time_dt
        account_age_years = delta.days / 365.25

        # Lógica de classificação de nível
        if account_age_years >= MIN_YEARS_FOR_ADULT_INFERENCE:
            level = "Veterano"
        elif account_age_years >= YEARS_FOR_INTERMEDIATE:
            level = "Intermediário"
        else:
            level = "Iniciante"

        user_info = {
            "user_id": data.get('id'),
            "display_name": data.get('displayName'),
            "location": data.get('locale', 'N/A'),
            "created_at": create_time_str,
            "account_age_years": round(account_age_years, 2),
            "level": level,
            "is_inferred_adult": account_age_years >= MIN_YEARS_FOR_ADULT_INFERENCE
        }
        processed_users.append(user_info)

    # Delay para evitar hitting rate limits agressivos em loops rápidos
    time.sleep(0.2)

import pandas as pd
df_users = pd.DataFrame(processed_users)
display(df_users.head(10))

🔍 Iniciando processamento de 25 usuários...

✅ Sucesso [GET]: https://apis.roblox.com/cloud/v2/users/947838656
✅ Sucesso [GET]: https://apis.roblox.com/cloud/v2/users/1027961385
✅ Sucesso [GET]: https://apis.roblox.com/cloud/v2/users/357829182
✅ Sucesso [GET]: https://apis.roblox.com/cloud/v2/users/244267273
✅ Sucesso [GET]: https://apis.roblox.com/cloud/v2/users/740396998
✅ Sucesso [GET]: https://apis.roblox.com/cloud/v2/users/731822913
✅ Sucesso [GET]: https://apis.roblox.com/cloud/v2/users/871184693
✅ Sucesso [GET]: https://apis.roblox.com/cloud/v2/users/262953699
✅ Sucesso [GET]: https://apis.roblox.com/cloud/v2/users/940534557
✅ Sucesso [GET]: https://apis.roblox.com/cloud/v2/users/970378427
✅ Sucesso [GET]: https://apis.roblox.com/cloud/v2/users/386824609
✅ Sucesso [GET]: https://apis.roblox.com/cloud/v2/users/747056849
✅ Sucesso [GET]: https://apis.roblox.com/cloud/v2/users/452079889
✅ Sucesso [GET]: https://apis.roblox.com/cloud/v2/users/519533783
✅ Sucesso [GET]: https://apis.

,user_id,display_name,location,created_at,account_age_years,level,is_inferred_adult
0,947838656,Scriptbloxian,en_us,2019-01-28T20:56:39.770Z,7.54,Intermediário,False
1,1027961385,Iamsmarterthanyahya3,en_us,2019-04-04T20:14:50.953Z,7.36,Intermediário,False
2,357829182,FallenGh,en_us,2017-08-02T17:39:17.880Z,9.03,Intermediário,False
3,244267273,kidsgivemesloppy,en_us,2017-02-20T05:17:52.727Z,9.48,Intermediário,False
4,740396998,Lloyd,en_us,2018-08-30T19:34:01.370Z,7.96,Intermediário,False
5,731822913,Kirill_pro300,ru_ru,2018-08-26T16:09:21.050Z,7.97,Intermediário,False
6,871184693,hunika201,en_us,2018-11-24T09:20:23.383Z,7.72,Intermediário,False
7,262953699,kevin2598,fr_fr,2017-03-14T15:33:27.687Z,9.42,Intermediário,False
8,940534557,furiouse_2,en_us,2019-01-23T01:56:45.160Z,7.56,Intermediário,False
9,970378427,Mid_Bacon,en_us,2019-02-16T16:54:47.510Z,7.49,Intermediário,False


### Exportação de Dados
Salvando o conjunto de dados processado em um arquivo CSV para uso externo.

In [20]:
# Exportando os usuários processados
save_df_to_csv(df_users, "usuarios_roblox_processados.csv")

✅ Arquivo 'usuarios_roblox_processados.csv' salvo com sucesso!


## First Hop